# VinAI Translate v3 training on Kaggle

This notebook supports fully offline training from `NLP_v3_frozen.zip`, local VinAI model snapshots, and an optional Linux wheelhouse attached as Kaggle Inputs. Offline mode blocks Hugging Face network access and loads weights only from `/kaggle/input`.

## 1. Kiểm tra GPU và môi trường

Thành công khi CUDA khả dụng và cell in đúng tên GPU.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # giữ effective batch giống Colab/nhóm
import subprocess, sys
subprocess.run(["nvidia-smi"], check=False)
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Kaggle Notebook settings.")
print("GPU dùng cho baseline:", torch.cuda.get_device_name(0))
print("Số GPU nhìn thấy bởi tiến trình:", torch.cuda.device_count())

## 2. Nhận diện project từ Kaggle Input

Kaggle có thể tự giải nén ZIP khi tạo Dataset. Cell này hỗ trợ cả project đã giải nén và file `.zip`, sau đó sao chép/giải nén sang `/kaggle/working`. Chỉ đặt `INPUT_NAME_HINT` khi notebook có nhiều Input.

In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, shutil, tempfile, zipfile, sys

INPUT_NAME_HINT = ''
input_root = Path('/kaggle/input')
if not input_root.is_dir():
    raise FileNotFoundError('Missing /kaggle/input; use Add Input first.')

def is_project_root(path):
    return ((path / 'requirements-training.txt').is_file()
            and (path / 'src/vinai_baseline/cli.py').is_file()
            and (path / 'configs/generated_v3').is_dir())

def find_project_roots(root):
    roots = [root] if is_project_root(root) else []
    roots += [p.parent for p in root.rglob('requirements-training.txt')
              if p.parent != root and is_project_root(p.parent)]
    return sorted(set(roots), key=lambda p: (len(p.parts), str(p)))

def normalized(value):
    return ''.join(ch for ch in str(value).casefold() if ch.isalnum())

def safe_extract(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    destination_root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        corrupt = archive.testzip()
        if corrupt: raise ValueError(f'Corrupt ZIP entry {corrupt}: {zip_path}')
        for member in archive.infolist():
            relative = PurePosixPath(member.filename.replace('\\', '/'))
            if relative.is_absolute() or '..' in relative.parts or not relative.parts:
                raise ValueError(f'Unsafe ZIP path: {member.filename}')
            target = destination.joinpath(*relative.parts)
            if not target.resolve().is_relative_to(destination_root):
                raise ValueError(f'Unsafe ZIP path: {member.filename}')
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open('wb') as output:
                    shutil.copyfileobj(source, output)

def zip_contains_project(path):
    try:
        with zipfile.ZipFile(path) as archive:
            names = {name.replace('\\', '/').lstrip('/') for name in archive.namelist()}
        required = ('requirements-training.txt', 'src/vinai_baseline/cli.py',
                    'artifacts/processed_clean_semantic_seed67_v3/manifest.json')
        return all(any(name.endswith(item) for name in names) for item in required)
    except zipfile.BadZipFile:
        return False

hint = normalized(INPUT_NAME_HINT)
def matches_hint(path): return not hint or hint in normalized(path)
direct_candidates = [p for p in find_project_roots(input_root) if matches_hint(p)]
project_zips = [p for p in sorted(input_root.rglob('*.zip'))
                if matches_hint(p) and zip_contains_project(p)]
working_root = Path(tempfile.mkdtemp(prefix='nlp_kaggle_', dir='/kaggle/working'))
if len(direct_candidates) == 1:
    PROJECT_DIR = working_root / 'project'
    shutil.copytree(direct_candidates[0], PROJECT_DIR)
    input_mode = 'extracted Kaggle project'
elif not direct_candidates and len(project_zips) == 1:
    extract_dir = working_root / 'project'
    safe_extract(project_zips[0], extract_dir)
    roots = find_project_roots(extract_dir)
    if len(roots) != 1:
        raise ValueError(f'Expected one project root, found: {roots}')
    PROJECT_DIR = roots[0]
    input_mode = f'ZIP: {project_zips[0]}'
else:
    raise ValueError(f'Expected one NLP_v3_frozen project input. direct={direct_candidates}, zips={project_zips}')

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path: sys.path.insert(0, str(PROJECT_DIR))
print('Input mode:', input_mode)
print('Project:', PROJECT_DIR)

## 3. Offline dependencies

With `OFFLINE_TRAINING=True`, pip uses only the attached Linux wheelhouse and cannot reach PyPI. PyTorch comes from the Kaggle GPU runtime.

In [ ]:
OFFLINE_TRAINING = True
WHEELHOUSE_HINT = 'vinai-training-wheelhouse'
requirement_file = (PROJECT_DIR / 'requirements-training-colab.txt'
                    if (PROJECT_DIR / 'requirements-training-colab.txt').exists()
                    else PROJECT_DIR / 'requirements-training.txt')

if OFFLINE_TRAINING:
    os.environ['HF_HUB_OFFLINE'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    os.environ['HF_DATASETS_OFFLINE'] = '1'
    os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
    os.environ['WANDB_DISABLED'] = 'true'
    wheel_dirs = sorted({p.parent for p in input_root.rglob('transformers-5.17.0-*.whl')})
    if not wheel_dirs:
        wheel_zips = [p for p in input_root.rglob('*.zip')
                      if normalized(WHEELHOUSE_HINT) in normalized(p)]
        if len(wheel_zips) != 1:
            raise FileNotFoundError(f'Expected one offline wheelhouse Input, found: {wheel_zips}')
        wheel_extract = working_root / 'wheelhouse'
        safe_extract(wheel_zips[0], wheel_extract)
        wheel_dirs = sorted({p.parent for p in wheel_extract.rglob('transformers-5.17.0-*.whl')})
    if len(wheel_dirs) != 1:
        raise ValueError(f'Expected one wheelhouse directory, found: {wheel_dirs}')
    WHEELHOUSE_DIR = wheel_dirs[0]
    install = [sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
               '--find-links', str(WHEELHOUSE_DIR), '-r', str(requirement_file)]
else:
    install = [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirement_file)]
subprocess.run(install, check=True)

from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version
expected_versions = {
    'transformers': '5.17.0', 'accelerate': '1.15.0', 'peft': '0.21.0',
    'sacrebleu': '2.6.0', 'sentencepiece': '0.2.2', 'protobuf': '6.33.6'}
actual_versions = {name: version(name) for name in expected_versions}
if actual_versions != expected_versions:
    raise RuntimeError(f'Dependency versions do not match: {actual_versions}')
try:
    torchao_version = version('torchao')
except PackageNotFoundError:
    torchao_version = None
if torchao_version and Version(torchao_version) <= Version('0.16.0'):
    print(f'Removing incompatible torchao {torchao_version}...')
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
print('Offline dependencies: PASSED', actual_versions)

## 4. Offline VinAI model inputs

Attach one model ZIP/folder for the selected direction, or attach both. The notebook validates local files and sets direction-specific paths; no Hugging Face token is used.

In [ ]:
import json

MODEL_SPECS = {
    'en2vi': {
        'folder': 'vinai-translate-en2vi-v2',
        'repo': 'vinai/vinai-translate-en2vi-v2',
        'revision': '1e47495b5dac16097b1d9e7b4133c27e1f40504f',
        'env': 'VINAI_EN2VI_MODEL_DIR'},
    'vi2en': {
        'folder': 'vinai-translate-vi2en-v2',
        'repo': 'vinai/vinai-translate-vi2en-v2',
        'revision': '4eb2219d447d4352861e71e371a6b6abcb0509c8',
        'env': 'VINAI_VI2EN_MODEL_DIR'},
}
required_model_files = {'config.json', 'generation_config.json',
                        'sentencepiece.bpe.model', 'model.safetensors'}

def valid_model_dirs(root, folder):
    candidates = []
    for path in root.rglob(folder):
        if path.is_dir() and required_model_files.issubset({p.name for p in path.iterdir() if p.is_file()}):
            candidates.append(path)
    return sorted(set(candidates), key=str)

def prepare_model_input(direction):
    spec = MODEL_SPECS[direction]
    candidates = valid_model_dirs(input_root, spec['folder'])
    if not candidates:
        model_zips = [p for p in input_root.rglob('*.zip')
                      if normalized(spec['folder']) in normalized(p)]
        if len(model_zips) == 1:
            destination = working_root / 'offline_models' / direction
            safe_extract(model_zips[0], destination)
            candidates = valid_model_dirs(destination, spec['folder'])
    if not candidates:
        return None
    if len(candidates) != 1:
        raise ValueError(f'Multiple {direction} model directories: {candidates}')
    model_dir = candidates[0]
    manifest_path = model_dir / 'offline_model_manifest.json'
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest.get('repo_id') != spec['repo'] or manifest.get('revision') != spec['revision']:
            raise ValueError(f'Wrong model manifest for {direction}: {manifest}')
        for name, metadata in manifest.get('files', {}).items():
            file_path = model_dir / name
            digest = hashlib.sha256()
            with file_path.open('rb') as handle:
                for block in iter(lambda: handle.read(1024 * 1024), b''):
                    digest.update(block)
            if digest.hexdigest() != metadata.get('sha256'):
                raise ValueError(f'Model checksum mismatch for {direction}: {name}')
    os.environ[spec['env']] = str(model_dir.resolve())
    return model_dir

LOCAL_MODEL_DIRS = {direction: prepare_model_input(direction) for direction in MODEL_SPECS}
print('Offline model inputs:', LOCAL_MODEL_DIRS)

## 5. Verify frozen v3 dataset

Training starts only from `NLP_v3_frozen.zip`. Semantic audit and freeze are completed in notebook 04 and are never rerun here.

In [ ]:
import json
V3_FROZEN_DIR = PROJECT_DIR / 'artifacts/processed_clean_semantic_seed67_v3'
V3_CONFIG_DIR = PROJECT_DIR / 'configs/generated_v3'
frozen_manifest_path = V3_FROZEN_DIR / 'manifest.json'
if not frozen_manifest_path.is_file():
    raise FileNotFoundError(f'Missing frozen v3 manifest: {frozen_manifest_path}')
frozen_manifest = json.loads(frozen_manifest_path.read_text(encoding='utf-8'))
if frozen_manifest.get('dataset_id') != 'legal_envi_clean_semantic_seed67_v3':
    raise ValueError(f'Wrong frozen dataset: {frozen_manifest.get("dataset_id")}')
if frozen_manifest.get('semantic_audit_only') is not False:
    raise ValueError('Audit-only data cannot be used for training')
print('Frozen v3: PASSED | counts:', frozen_manifest['counts'])

## 6. Ch?n th? nghi?m v? resume

Notebook ch? d?ng dataset v3 ?? semantic audit v? freeze trong `artifacts/processed_clean_semantic_seed67_v3`, c?ng config trong `configs/generated_v3`. Gi? `RUN_FULL=False` ?? smoke tr??c. Khi resume t? output c?a phi?n Kaggle c?, attach output ?? l?m Input, ?i?n `RUN_ID_OVERRIDE`, ??t `RESUME_FROM_CHECKPOINT='last'`, `RUN_FULL=True` v? tr? `RESUME_RUN_SOURCE` t?i ??ng th? m?c run trong `/kaggle/input`.

In [ ]:
import json, re
from datetime import datetime, timezone
DATASET_PROFILE = "v3_frozen"  # profile duy nh?t ???c h? tr?
DIRECTION = "en2vi"
MODE = "lora"  # zero_shot | lora | full
CONFIG_OVERRIDE = ""  # chỉ dùng cho config tùy chỉnh; bình thường để trống
RUN_FULL = False
MEMBER_TAG = "thuan"
EXPERIMENT_TAG = "lora_v3_semantic070_offline"
RUN_ID_OVERRIDE = ""
RESUME_FROM_CHECKPOINT = ""
RESUME_RUN_SOURCE = ""  # ví dụ /kaggle/input/previous-run/.../<RUN_ID>

if DATASET_PROFILE not in {"v2", "v3_frozen"}: raise ValueError("DATASET_PROFILE không hợp lệ")
if DIRECTION not in {"en2vi", "vi2en"}: raise ValueError("DIRECTION không hợp lệ")
if MODE not in {"zero_shot", "lora", "full"}: raise ValueError("MODE không hợp lệ")
for label, value in {"MEMBER_TAG": MEMBER_TAG, "EXPERIMENT_TAG": EXPERIMENT_TAG}.items():
    if not re.fullmatch(r"[a-z0-9_-]+", value): raise ValueError(f"{label} không hợp lệ")
if RUN_ID_OVERRIDE and not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_ID_OVERRIDE):
    raise ValueError("RUN_ID_OVERRIDE không hợp lệ")
if RESUME_FROM_CHECKPOINT and (not RUN_ID_OVERRIDE or not RUN_FULL or MODE == "zero_shot"):
    raise ValueError("Resume cần RUN_ID_OVERRIDE, RUN_FULL=True và mode lora/full")
default_config_dir = PROJECT_DIR / "configs/generated_v3"
CONFIG = PROJECT_DIR / CONFIG_OVERRIDE if CONFIG_OVERRIDE else default_config_dir / f"{DIRECTION}_{MODE}.json"
if not CONFIG.is_file():
    hint = ("Hãy dùng NLP_colab.zip có dataset v2." if DATASET_PROFILE == "v2" else
            "Hãy chạy semantic audit/freeze ở phần 5 để sinh config v3.")
    raise FileNotFoundError(f"Không tìm thấy config: {CONFIG}. {hint}")
config_preview = json.loads(CONFIG.read_text(encoding="utf-8"))
expected_training_mode = "none" if MODE == "zero_shot" else MODE
if config_preview["direction"] != DIRECTION or config_preview["training"]["mode"] != expected_training_mode:
    raise ValueError("DIRECTION/MODE không khớp với CONFIG_OVERRIDE")
DATASET_ID = config_preview["data"]["dataset_id"]
expected_dataset_id = "legal_envi_clean_semantic_seed67_v3"
if DATASET_ID != expected_dataset_id:
    raise ValueError(f"DATASET_PROFILE={DATASET_PROFILE} yêu cầu dataset_id={expected_dataset_id}, "
                     f"nhưng config đang dùng {DATASET_ID}")
TRAIN_SEED = config_preview["seed"]
DATA_DIR = PROJECT_DIR / Path(config_preview["data"]["manifest_file"]).parent
RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_ID = RUN_ID_OVERRIDE or f"{DATASET_ID}__{DIRECTION}__{MODE}__seed{TRAIN_SEED}__{MEMBER_TAG}__{EXPERIMENT_TAG}__{RUN_STAMP}"
OUTPUT_DIR = Path("/kaggle/working/NLP_runs") / DATASET_ID / DIRECTION / MODE / RUN_ID
if RESUME_FROM_CHECKPOINT:
    source = Path(RESUME_RUN_SOURCE)
    if not source.is_dir() or source.name != RUN_ID:
        raise FileNotFoundError("RESUME_RUN_SOURCE phải là đúng thư mục RUN_ID đã attach vào Kaggle Input")
    shutil.copytree(source, OUTPUT_DIR, dirs_exist_ok=True)
manifest_path = PROJECT_DIR / config_preview["data"]["manifest_file"]
if not manifest_path.is_file():
    raise FileNotFoundError(f"Thiếu manifest của {DATASET_PROFILE}: {manifest_path}")
manifest_preview = json.loads(manifest_path.read_text(encoding="utf-8"))
if manifest_preview.get("dataset_id") != DATASET_ID:
    raise ValueError("dataset_id trong config và manifest không khớp")
if manifest_preview.get("split_contract", {}).get("seed") != 67 or TRAIN_SEED != 67:
    raise ValueError("Split seed và training seed đều phải là 67")
if manifest_preview.get("semantic_audit_only"):
    raise ValueError("Semantic audit chỉ để duyệt score, không được dùng để train")
required = [DATA_DIR / f"segments_dev_{DIRECTION}.jsonl"]
if MODE != "zero_shot": required.append(DATA_DIR / f"segments_train_{DIRECTION}.jsonl")
missing = [str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError("Thiếu dữ liệu: " + ", ".join(missing))
if OFFLINE_TRAINING:
    model_spec = MODEL_SPECS[DIRECTION]
    local_model = LOCAL_MODEL_DIRS.get(DIRECTION)
    if local_model is None:
        raise FileNotFoundError(
            f"Missing offline model for {DIRECTION}. Attach {model_spec['folder']} and rerun from section 4.")
    if (config_preview.get('model_name') != model_spec['repo']
            or config_preview.get('model_revision') != model_spec['revision']):
        raise ValueError('Config model identity does not match the pinned offline snapshot')
    print('Local base model:', local_model)
training = config_preview["training"]
print("Dataset profile:", DATASET_PROFILE, "| dataset ID:", DATASET_ID)
print("Config:", CONFIG)
print("Run ID:", RUN_ID)
print("Output:", OUTPUT_DIR)
print("Profile: train batch", training.get("per_device_train_batch_size"),
      "x accumulation", training.get("gradient_accumulation_steps"),
      "| eval batch", training.get("per_device_eval_batch_size"),
      "| checkpoint samples", training.get("checkpoint_eval_samples"))

## 7. Audit và hàm chạy có log

Log được stream trực tiếp và lưu trong thư mục `logs/` của run. Tiếp theo cần chạy zero-shot 256 mẫu ở phần 7b trước khi train.

In [ ]:
from time import perf_counter
def format_duration(seconds):
    total = round(seconds); hours, rem = divmod(total, 3600); minutes, seconds = divmod(rem, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

def run_cli(command, *extra):
    args = [sys.executable, "-u", "-m", "src.vinai_baseline.cli", command,
            "--config", str(CONFIG), "--output-dir", str(OUTPUT_DIR), *map(str, extra)]
    started = perf_counter()
    print(f"\n[{datetime.now().astimezone().isoformat(timespec='seconds')}] BẮT ĐẦU")
    print("RUN:", " ".join(args), flush=True)
    env = os.environ.copy(); env["PYTHONUNBUFFERED"] = "1"
    # Đặt log trong chính run để ZIP kết quả luôn chứa audit/train/verify log.
    log_dir = Path(OUTPUT_DIR) / "logs"; log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"{command}_{datetime.now().strftime('%Y%m%dT%H%M%S%f')}.log"
    lines = []
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(args, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                                   errors="replace", bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True); log_file.write(line); log_file.flush(); lines.append(line.rstrip())
        return_code = process.wait()
    elapsed = perf_counter() - started
    if return_code:
        tail = "\n".join(lines[-80:]) or "(không có log)"
        raise RuntimeError(f"Lệnh {command} lỗi code {return_code} sau {format_duration(elapsed)}\n{tail}\nLog: {log_path}") from None
    print("HOÀN THÀNH sau", format_duration(elapsed), "| Log:", log_path)
    return elapsed

collator_source = (PROJECT_DIR / "src/vinai_baseline/data.py").read_text(encoding="utf-8")
if "prepare_decoder_input_ids_from_labels" not in collator_source:
    raise RuntimeError("ZIP cũ, chưa có bản sửa labels/logits LoRA")
if not (PROJECT_DIR / "src/vinai_baseline/language.py").is_file():
    raise RuntimeError("ZIP cũ: thiếu bản sửa language control")
print("Preflight LoRA collator: OK")
run_cli("audit")

## 7b. Zero-shot 256 mẫu — bắt buộc trước khi train

Chạy model gốc với language control đã sửa trên tập dev cố định, cân bằng category/document. Cell ghi metrics, predictions và quality gate vào Output. Gate v2 chỉ chặn lỗi sinh nghiêm trọng (rỗng, lặp/token collapse, quá dài hoặc sai ngôn ngữ); bản dịch quá ngắn được ghi ở `quality.warnings` để đo lại sau LoRA. Mã thoát 2 nghĩa là hard gate fail. Đây là kiểm tra suy biến, không bảo đảm dịch chính xác mọi câu.

In [ ]:
from src.vinai_baseline.config import ExperimentConfig
from src.vinai_baseline.preflight_cli import require_zero_shot_gate
DIAGNOSTIC_ATTEMPT = '01'  # tăng lên 02 nếu lần trước bị ngắt; giữ report cũ để đối chiếu
ZERO_GATE_DIR = Path('/kaggle/working/diagnostics') / (RUN_ID + '__zero256_' + DIAGNOSTIC_ATTEMPT)
ZERO_GATE_REPORT = ZERO_GATE_DIR / 'gate_report.json'
def require_preflight():
    if not ZERO_GATE_REPORT.is_file():
        raise RuntimeError('Hãy chạy zero-shot 256 mẫu ở phần 7b trước')
    require_zero_shot_gate(ExperimentConfig.from_json(CONFIG), ZERO_GATE_REPORT)
if not ZERO_GATE_REPORT.exists():
    command = [sys.executable, '-u', '-m', 'src.vinai_baseline.preflight_cli',
               '--config', str(CONFIG), '--output-dir', str(ZERO_GATE_DIR), '--samples', '256']
    result = subprocess.run(command, cwd=PROJECT_DIR)
    if result.returncode:
        if ZERO_GATE_REPORT.is_file():
            failed_gate = json.loads(ZERO_GATE_REPORT.read_text(encoding='utf-8'))
            print('Zero-shot hard failures:')
            for failure in failed_gate.get('quality', {}).get('failures', []):
                print(json.dumps(failure, ensure_ascii=False, indent=2))
        raise RuntimeError(f'Zero-shot chưa pass (code={result.returncode}). Xem {ZERO_GATE_DIR}; không train tiếp.')
require_preflight()
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
shutil.copytree(ZERO_GATE_DIR, Path(OUTPUT_DIR) / 'preflight_zero256', dirs_exist_ok=True)
gate_data = json.loads(ZERO_GATE_REPORT.read_text(encoding='utf-8'))
print('Zero-shot gate PASSED:', ZERO_GATE_REPORT)
print('Cảnh báo cần cải thiện bằng LoRA:', gate_data['quality'].get('warning_records', 0), '/', gate_data['quality']['evaluated_records'])


## 7c. Nạp hoặc đánh giá lại adapter cũ — không train

Cell tự tìm `best_model` đã được Kaggle giải nén hoặc ZIP kết quả trong `/kaggle/input`, `PROJECT_DIR/attached_runs`. Với run đã chấm full-dev, đặt `LOAD_COMPLETED_ADAPTER_FOR_SWEEP=True` để nạp thẳng adapter cho phần 11 mà không chạy lại train. Muốn chấm diagnostic 256 mẫu thì đặt thêm `RUN_OLD_ADAPTER_CHECK=True`; nhánh này vẫn yêu cầu gate zero-shot ở 7b.

In [ ]:
RUN_OLD_ADAPTER_CHECK = False
LOAD_COMPLETED_ADAPTER_FOR_SWEEP = False
OLD_ADAPTER_DIR = ''  # để trống để tự tìm best_model hoặc ZIP result
OLD_RESULT_HINT = DIRECTION  # đổi nếu có nhiều ZIP kết quả cùng chiều
CHECKED_ADAPTER_DIR = None
def locate_completed_adapter():
    if OLD_ADAPTER_DIR:
        candidates = [Path(OLD_ADAPTER_DIR)]
    else:
        roots = [Path('/kaggle/input'), PROJECT_DIR / 'attached_runs']
        candidates = []
        for root in roots:
            if root.is_dir():
                candidates.extend(path.parent for path in root.rglob('adapter_config.json')
                                  if path.parent.name == 'best_model')
        candidates = sorted(set(candidates), key=str)
    if not candidates:
        zip_candidates = []
        for root in [Path('/kaggle/input'), PROJECT_DIR / 'attached_runs']:
            if root.is_dir():
                zip_candidates.extend(path for path in root.rglob('*.zip')
                    if OLD_RESULT_HINT.casefold() in path.name.casefold()
                    and 'result' in path.name.casefold())
        zip_candidates = sorted(set(zip_candidates), key=str)
        if len(zip_candidates) != 1:
            raise FileNotFoundError(f'Không chọn được đúng một ZIP result: {zip_candidates}')
        result_zip = zip_candidates[0]
        destination = Path('/kaggle/working/recovered_runs') / result_zip.stem
        destination.mkdir(parents=True, exist_ok=True)
        destination_root = destination.resolve()
        with zipfile.ZipFile(result_zip) as archive:
            for member in archive.infolist():
                relative = PurePosixPath(member.filename.replace('\\', '/'))
                if relative.is_absolute() or '..' in relative.parts or not relative.parts:
                    raise ValueError(f'ZIP result có đường dẫn không an toàn: {member.filename}')
                if relative.parts[0] not in {'best_model', 'reports'} and relative.name not in {'resolved_config.json', 'run_manifest.json'}:
                    continue
                target = destination.joinpath(*relative.parts)
                if not target.resolve().is_relative_to(destination_root):
                    raise ValueError(f'ZIP result có đường dẫn không an toàn: {member.filename}')
                if member.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                else:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with archive.open(member) as source, target.open('wb') as output:
                        shutil.copyfileobj(source, output)
        candidates = [destination / 'best_model']
    if len(candidates) != 1:
        raise ValueError(f'Tìm thấy nhiều best_model; hãy đặt OLD_ADAPTER_DIR: {candidates}')
    adapter_path = candidates[0]
    if not (adapter_path / 'adapter_config.json').is_file():
        raise FileNotFoundError(f'Thiếu adapter_config.json: {adapter_path}')
    run_root = adapter_path.parent
    resolved_path = run_root / 'resolved_config.json'
    metrics_path = run_root / 'reports/validation_metrics.json'
    if resolved_path.is_file():
        old_config = json.loads(resolved_path.read_text(encoding='utf-8'))
        if old_config.get('direction') != DIRECTION or old_config.get('model_name') != config_preview.get('model_name'):
            raise ValueError('Adapter cũ không khớp direction/base model hiện tại')
    if LOAD_COMPLETED_ADAPTER_FOR_SWEEP:
        if not metrics_path.is_file():
            raise FileNotFoundError('Chế độ completed adapter yêu cầu reports/validation_metrics.json')
        old_metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
        if int(old_metrics.get('evaluated_records', 0)) < 256:
            raise ValueError('Run cũ chưa có đánh giá đủ lớn để dùng trực tiếp cho sweep')
    return adapter_path
if RUN_OLD_ADAPTER_CHECK or LOAD_COMPLETED_ADAPTER_FOR_SWEEP:
    adapter_path = locate_completed_adapter()
    if LOAD_COMPLETED_ADAPTER_FOR_SWEEP:
        CHECKED_ADAPTER_DIR = adapter_path
        print('Đã nạp completed adapter cho sweep:', CHECKED_ADAPTER_DIR)
if RUN_OLD_ADAPTER_CHECK:
    require_preflight()
    if MODE != 'lora': raise ValueError('Chọn MODE=lora để kiểm tra adapter cũ')
    adapter_output = Path('/kaggle/working/diagnostics') / (RUN_ID + '__old_adapter_' + datetime.now(timezone.utc).strftime('%H%M%S%f'))
    result = subprocess.run([sys.executable, '-u', '-m', 'src.vinai_baseline.preflight_cli',
        '--config', str(CONFIG), '--saved-model-dir', str(adapter_path),
        '--output-dir', str(adapter_output), '--samples', '256'], cwd=PROJECT_DIR)
    if result.returncode not in (0, 2): raise RuntimeError('Lỗi chạy diagnostic; xem traceback phía trên')
    shutil.copytree(adapter_output, Path(OUTPUT_DIR) / adapter_output.name, dirs_exist_ok=True)
    if result.returncode == 0:
        CHECKED_ADAPTER_DIR = adapter_path
    print('Adapter cũ:', 'PASSED' if result.returncode == 0 else 'FAILED — train mới, không resume', adapter_output)
elif not LOAD_COMPLETED_ADAPTER_FOR_SWEEP:
    print('Chưa nạp adapter cũ; dùng phần 8 nếu muốn train một run mới.')


## 8. Smoke test

LoRA/full dùng 32 mẫu train và 16 validation; zero-shot dùng 16 validation. Smoke chỉ xác minh pipeline, không kết luận chất lượng.

In [ ]:
SMOKE_PASSED = False
if MODE != "zero_shot": require_preflight()
COMMAND = "evaluate" if MODE == "zero_shot" else "train"
if RESUME_FROM_CHECKPOINT:
    RESULT_DIR = OUTPUT_DIR
    print("Bỏ qua smoke vì đang resume.")
else:
    SMOKE_OUTPUT = Path("/kaggle/working/vinai_smoke") / RUN_ID
    full_output = OUTPUT_DIR
    try:
        OUTPUT_DIR = SMOKE_OUTPUT
        if MODE == "zero_shot":
            run_cli(COMMAND, "--max-validation-samples", "16")
        else:
            run_cli(COMMAND, "--max-train-samples", "32", "--max-validation-samples", "16")
            run_cli("verify", "--max-validation-samples", "16")
        RESULT_DIR = SMOKE_OUTPUT
    finally:
        OUTPUT_DIR = full_output
    SMOKE_PASSED = True
    SMOKE_CONFIG = str(CONFIG)
    SMOKE_RUN_ID = RUN_ID
    print("Smoke hoàn thành:", RESULT_DIR)

## 9. Full run

Sau khi smoke pass, đổi `START_FULL_NOW=True` ngay trong cell dưới rồi chạy cell này; không chạy lại cell cấu hình nên `RUN_ID` được giữ nguyên. Khi resume, `RUN_FULL=True` từ cell cấu hình cũng tự kích hoạt full run. Output nằm trong `/kaggle/working/NLP_runs`; hãy **Save Version** hoặc tải ZIP kết quả ở cell cuối trước khi phiên kết thúc.

In [ ]:
START_FULL_NOW = False  # đổi True sau khi smoke pass; chỉ chạy lại cell này
if START_FULL_NOW:
    RUN_FULL = True
if RUN_FULL:
    if MODE != "zero_shot": require_preflight()
    if not RESUME_FROM_CHECKPOINT and not (globals().get("SMOKE_PASSED", False)
            and globals().get("SMOKE_CONFIG") == str(CONFIG)
            and globals().get("SMOKE_RUN_ID") == RUN_ID):
        raise RuntimeError("Hãy chạy smoke thành công cho đúng run này trước full run")
    COMMAND = "evaluate" if MODE == "zero_shot" else "train"
    resume_args = (["--resume-from-checkpoint", RESUME_FROM_CHECKPOINT] if RESUME_FROM_CHECKPOINT else [])
    run_cli(COMMAND, *resume_args)
    RESULT_DIR = OUTPUT_DIR  # giữ kết quả full ngay cả khi verify phát hiện lỗi chất lượng
    if MODE != "zero_shot": run_cli("verify")
    RESULT_DIR = OUTPUT_DIR
else:
    print("Chưa chạy full; đang dùng kết quả smoke:", RESULT_DIR)

## 10. Xem metric, ví dụ dịch và đóng gói output

Cell kiểm tra đủ prediction, hiển thị metric/mẫu dịch và tạo một ZIP trong Kaggle Output để tải về hoặc dùng làm Input cho phiên resume.

In [ ]:
import random, pandas as pd
from IPython.display import display
RESULT_DIR = Path(RESULT_DIR)
metrics_path = RESULT_DIR / "reports/validation_metrics.json"
if not metrics_path.exists(): raise FileNotFoundError("Run chưa có validation_metrics.json")
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
quality_path = RESULT_DIR / 'reports/quality_gate.json'
if quality_path.is_file():
    quality = json.loads(quality_path.read_text(encoding='utf-8'))
    print('Full evaluation quality gate:', quality['status'], '| hard failures:', quality['failed_records'], '| warnings:', quality.get('warning_records', 0))
    if quality['status'] != 'passed': print('Cần duyệt quality_gate.json; chưa dùng run này làm baseline đạt chất lượng.')
prediction_path = RESULT_DIR / "predictions/validation_predictions.jsonl"
predictions = [json.loads(line) for line in prediction_path.read_text(encoding="utf-8").splitlines() if line.strip()]
expected = int(metrics["evaluated_records"])
if len(predictions) != expected: raise RuntimeError(f"Prediction {len(predictions)}/{expected}")
def metric(name): return metrics.get(f"validation_{name}", metrics.get(name))
summary = [(name, metric(name)) for name in ["bleu", "chrf", "empty_output_rate",
           "length_ratio_mean", "length_ratio_p95", "length_ratio_over_3_rate",
           "repeated_trigram_rate", "dominant_token_over_20_rate", "number_preservation_f1",
           "source_number_preservation_f1", "reference_preservation_f1",
           "legal_identifier_preservation_f1", "source_legal_identifier_preservation_f1", "runtime"]]
summary.append(("peak_vram_mb", metrics.get("peak_vram_mb")))
display(pd.DataFrame(summary, columns=["metric", "value"]))
indices = random.Random(67).sample(range(len(predictions)), min(5, len(predictions)))
display(pd.DataFrame([{k: predictions[i][k] for k in ["segment_id", "source", "prediction", "target"]} for i in indices]))
is_full_result = RESULT_DIR.resolve() == Path(OUTPUT_DIR).resolve()
archive_suffix = "result" if is_full_result else "smoke_result"
archive_base = Path("/kaggle/working") / f"{RUN_ID}_{archive_suffix}"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=RESULT_DIR)
if not is_full_result:
    print("LƯU Ý: đây chỉ là artifact smoke 32/16 mẫu, chưa phải kết quả full run.")
print("Artifact đầy đủ:", len(predictions), "prediction")
print("ZIP kết quả:", archive_path)

## 11. Generation sweep trên adapter đã pass (không train lại)

Nhánh khuyến nghị cho adapter epoch 2: chạy 7b và 7c, sau đó có thể bỏ qua 8–10 và chạy cell này. Sweep dùng đúng 256 mẫu của diagnostic để thử decoding; không thay đổi trọng số. `CHECKED_ADAPTER_DIR` chỉ được đặt khi adapter ở 7c đã pass. Raw repeated-trigram của văn bản luật có thể cao dù không suy biến, nên việc chọn vẫn phải xem BLEU/chrF++, bảo toàn số/tham chiếu/mã pháp lý và quality gate ở bước full-dev.

In [ ]:
import pandas as pd
from IPython.display import display
RUN_GENERATION_SWEEP = False  # đổi True sau khi adapter epoch 2 pass ở 7c
SWEEP_BATCH_SIZE = 8  # an toàn trên T4; 16 nhanh hơn nếu VRAM cho phép
SWEEP_CANDIDATES = [
    'baseline:2:1.0:1.0:3:none',
    'beam4:4:1.0:1.0:3:none',
    'longer:2:1.05:1.0:3:none',
    'longer110:2:1.10:1.0:3:none',
    'beam4longer:4:1.10:1.0:3:none',
    'repeat103:2:1.0:1.03:3:none',
]
if RUN_GENERATION_SWEEP:
    adapter_for_sweep = globals().get('CHECKED_ADAPTER_DIR')
    if adapter_for_sweep is None:
        candidate = Path(globals().get('RESULT_DIR', '')) / 'best_model'
        adapter_for_sweep = candidate if (candidate / 'adapter_config.json').is_file() else None
    if adapter_for_sweep is None or not (Path(adapter_for_sweep) / 'adapter_config.json').is_file():
        raise RuntimeError('Hãy chạy 7c với adapter epoch 2 và bảo đảm gate PASSED trước sweep.')
    SWEEP_ADAPTER_DIR = Path(adapter_for_sweep)
    SWEEP_DIR = Path('/kaggle/working/generation_sweeps') / f"epoch2__{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
    sweep_command = [sys.executable, '-u', '-m', 'src.vinai_baseline.sweep_cli',
        '--config', str(CONFIG), '--saved-model-dir', str(SWEEP_ADAPTER_DIR),
        '--output-dir', str(SWEEP_DIR), '--samples', '256',
        '--batch-size', str(SWEEP_BATCH_SIZE),
        '--maximum-repeated-trigram-rate', '100']
    for candidate in SWEEP_CANDIDATES:
        sweep_command.extend(['--candidate', candidate])
    subprocess.run(sweep_command, cwd=PROJECT_DIR, check=True)
    SWEEP_REPORT = SWEEP_DIR / 'sweep_summary.json'
    sweep = json.loads(SWEEP_REPORT.read_text(encoding='utf-8'))
    rows = []
    for candidate in sweep['candidates']:
        metric = candidate['metrics']
        rows.append({'name': candidate['name'], 'BLEU': metric['bleu'], 'chrF++': metric['chrf'],
            'length_mean': metric['length_ratio_mean'], 'length_p95': metric['length_ratio_p95'],
            'over_3_rate': metric['length_ratio_over_3_rate'],
            'dominant_over_20': metric['dominant_token_over_20_rate'],
            'number_F1': metric['number_preservation_f1'],
            'reference_F1': metric['reference_preservation_f1'],
            'legal_identifier_F1': metric['legal_identifier_preservation_f1'],
            'runtime_seconds': metric['runtime_seconds']})
    SWEEP_TABLE = pd.DataFrame(rows).sort_values(['chrF++', 'BLEU'], ascending=False)
    display(SWEEP_TABLE)
    print('Đề xuất theo chrF++/BLEU:', sweep['recommended'])
    print('Đọc bảng rồi điền SELECTED_GENERATION_NAME ở phần 12. Report:', SWEEP_REPORT)
else:
    print('Bỏ qua sweep; đổi RUN_GENERATION_SWEEP=True sau khi 7c pass.')

## 12. Chốt decoding và đánh giá toàn bộ 1.821 mẫu dev

Chỉ chạy sau phần 11. Điền đúng tên candidate từ bảng sweep, rồi bật `RUN_FULL_ADAPTER_EVAL=True`. Cell tạo một config riêng trong `/kaggle/working`, đánh giá adapter đã chọn trên toàn bộ dev sau token audit và đóng gói report/predictions. Đây vẫn là inference, hoàn toàn không train lại. Trên T4 có thể mất khoảng 45–60 phút.

In [ ]:
SELECTED_GENERATION_NAME = ''  # baseline | beam4 | longer | longer110 | beam4longer | repeat103
RUN_FULL_ADAPTER_EVAL = False
FULL_EVAL_BATCH_SIZE = 8
if RUN_FULL_ADAPTER_EVAL:
    if not SELECTED_GENERATION_NAME:
        raise ValueError('Hãy điền SELECTED_GENERATION_NAME từ bảng sweep.')
    if 'SWEEP_REPORT' not in globals() or not Path(SWEEP_REPORT).is_file():
        raise RuntimeError('Hãy chạy generation sweep ở phần 11 trước.')
    sweep = json.loads(Path(SWEEP_REPORT).read_text(encoding='utf-8'))
    matches = [item for item in sweep['candidates'] if item['name'] == SELECTED_GENERATION_NAME]
    if len(matches) != 1:
        raise ValueError(f'Candidate không tồn tại: {SELECTED_GENERATION_NAME}')
    winner = matches[0]
    selected_config = json.loads(Path(CONFIG).read_text(encoding='utf-8'))
    selected_config['generation'] = winner['generation']
    SELECTED_CONFIG = Path('/kaggle/working') / f'{DIRECTION}_epoch2_{SELECTED_GENERATION_NAME}.json'
    SELECTED_CONFIG.write_text(json.dumps(selected_config, ensure_ascii=False, indent=2), encoding='utf-8')
    FULL_ADAPTER_DIR = Path('/kaggle/working/diagnostics') / f"epoch2_full1821__{SELECTED_GENERATION_NAME}__{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
    command = [sys.executable, '-u', '-m', 'src.vinai_baseline.preflight_cli',
        '--config', str(SELECTED_CONFIG), '--saved-model-dir', str(SWEEP_ADAPTER_DIR),
        '--output-dir', str(FULL_ADAPTER_DIR), '--samples', '1821',
        '--batch-size', str(FULL_EVAL_BATCH_SIZE)]
    result = subprocess.run(command, cwd=PROJECT_DIR)
    if result.returncode not in (0, 2):
        raise RuntimeError(f'Full adapter evaluation lỗi kỹ thuật, code={result.returncode}')
    FULL_ADAPTER_REPORT = FULL_ADAPTER_DIR / 'gate_report.json'
    full_report = json.loads(FULL_ADAPTER_REPORT.read_text(encoding='utf-8'))
    quality = full_report['quality']; metric = full_report['metrics']
    display(pd.DataFrame([{'BLEU': metric['bleu'], 'chrF++': metric['chrf'],
        'length_mean': metric['length_ratio_mean'], 'number_F1': metric['number_preservation_f1'],
        'reference_F1': metric['reference_preservation_f1'],
        'legal_identifier_F1': metric['legal_identifier_preservation_f1'],
        'hard_failures': quality['failed_records'],
        'warnings': quality.get('warning_records', 0)}]))
    archive_path = shutil.make_archive(str(FULL_ADAPTER_DIR), 'zip', root_dir=FULL_ADAPTER_DIR)
    print('Gate:', quality['status'])
    print('Report:', FULL_ADAPTER_REPORT)
    print('ZIP để tải về:', archive_path)
    if result.returncode == 2:
        print('CẢNH BÁO: hard quality gate fail; đọc failures trước khi dùng adapter.')
else:
    print('Sau khi xem bảng phần 11: điền candidate, đổi RUN_FULL_ADAPTER_EVAL=True và chạy lại cell này.')

## 13. Public-test final evaluation — không train, không chọn lại config

Chỉ bật sau khi adapter và generation config đã được chốt hoàn toàn trên dev. Cell dịch đủ 1.211 public-test record theo pipeline document với chunk nguồn tối đa 448 token, tính metric bằng reference public, rồi xuất prediction/report/ZIP riêng cho từng chiều. Không dùng kết quả này để quay lại chọn checkpoint hay decoding. Với direction đang chạy, cell tự dùng `SELECTED_CONFIG` và `SWEEP_ADAPTER_DIR`; chiều còn lại phải khai báo đường dẫn config/adapter đã chốt trong hai mapping bên dưới.

In [ ]:
RUN_PUBLIC_TEST_FINAL = False
PUBLIC_TEST_DIRECTIONS = [DIRECTION]  # hoặc ['en2vi', 'vi2en'] khi đã có cả hai model
PUBLIC_TEST_BATCH_SIZE = 8
PUBLIC_TEST_MAX_SOURCE_TOKENS = 448
PUBLIC_TEST_CONFIGS = {
    'en2vi': '',  # đường dẫn selected config; để trống nếu đây là DIRECTION hiện tại
    'vi2en': '',
}
PUBLIC_TEST_ADAPTER_DIRS = {
    'en2vi': '',  # đường dẫn best_model; để trống nếu đây là DIRECTION hiện tại
    'vi2en': '',
}
if RUN_PUBLIC_TEST_FINAL:
    invalid = set(PUBLIC_TEST_DIRECTIONS) - {'en2vi', 'vi2en'}
    if invalid or not PUBLIC_TEST_DIRECTIONS:
        raise ValueError(f'PUBLIC_TEST_DIRECTIONS không hợp lệ: {PUBLIC_TEST_DIRECTIONS}')
    result_rows = []
    PUBLIC_TEST_OUTPUTS = {}
    for public_direction in PUBLIC_TEST_DIRECTIONS:
        config_value = PUBLIC_TEST_CONFIGS.get(public_direction, '')
        adapter_value = PUBLIC_TEST_ADAPTER_DIRS.get(public_direction, '')
        if not config_value and public_direction == DIRECTION:
            config_value = globals().get('SELECTED_CONFIG', '')
        if not adapter_value and public_direction == DIRECTION:
            adapter_value = globals().get('SWEEP_ADAPTER_DIR') or globals().get('CHECKED_ADAPTER_DIR')
        selected_config_path = Path(config_value) if config_value else None
        selected_adapter_path = Path(adapter_value) if adapter_value else None
        if selected_config_path is None or not selected_config_path.is_file():
            raise FileNotFoundError(f'Thiếu selected config cho {public_direction}: {config_value}')
        if selected_adapter_path is None or not (selected_adapter_path / 'adapter_config.json').is_file():
            raise FileNotFoundError(f'Thiếu best_model cho {public_direction}: {adapter_value}')
        selected_payload = json.loads(selected_config_path.read_text(encoding='utf-8'))
        if selected_payload.get('direction') != public_direction:
            raise ValueError(f'Config không khớp direction {public_direction}: {selected_config_path}')
        public_output = Path('/kaggle/working/public_test_final') / (
            f"{public_direction}__{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}")
        command = [sys.executable, '-u', '-m', 'src.vinai_baseline.public_test_cli',
            '--config', str(selected_config_path),
            '--saved-model-dir', str(selected_adapter_path),
            '--output-dir', str(public_output),
            '--batch-size', str(PUBLIC_TEST_BATCH_SIZE),
            '--maximum-source-tokens', str(PUBLIC_TEST_MAX_SOURCE_TOKENS)]
        result = subprocess.run(command, cwd=PROJECT_DIR)
        if result.returncode not in (0, 2):
            raise RuntimeError(f'Public test {public_direction} lỗi kỹ thuật, code={result.returncode}')
        report_path = public_output / 'public_test_report.json'
        report = json.loads(report_path.read_text(encoding='utf-8'))
        archive_path = shutil.make_archive(str(public_output), 'zip', root_dir=public_output)
        PUBLIC_TEST_OUTPUTS[public_direction] = {
            'directory': public_output, 'report': report_path, 'zip': Path(archive_path)}
        metric = report['metrics']; quality = report['quality']
        result_rows.append({'direction': public_direction, 'records': report['records'],
            'chunks': report['source_chunks'], 'BLEU': metric['bleu'], 'chrF++': metric['chrf'],
            'length_mean': metric['length_ratio_mean'],
            'number_F1': metric['number_preservation_f1'],
            'reference_F1': metric['reference_preservation_f1'],
            'legal_identifier_F1': metric['legal_identifier_preservation_f1'],
            'hard_failures': quality['failed_records'],
            'warnings': quality.get('warning_records', 0), 'zip': archive_path})
    PUBLIC_TEST_SUMMARY = pd.DataFrame(result_rows)
    display(PUBLIC_TEST_SUMMARY)
    print('Đây là final public-test report; không dùng bảng này để chọn lại model/config.')
else:
    print('Chỉ đặt RUN_PUBLIC_TEST_FINAL=True sau khi đã chốt adapter và decoding trên dev.')
